# True Parameters at 1 PA Model

```python
m1 = 1e6
m2 = 10
a = 0.8 
e0 = 0.4 
xI0 = 1.0
dist = 0.4

qS = xp.pi/4
phiS = 1.0
qK = 1
phiK = xp.pi/3

Phi_phi0 = 0.9
Phi_theta0 = 0.5
Phi_r0 = 0.4

dt = 10.0
T = 1.0

chi2 = 0.0

dev_0_p = 0.0
dev_0_e = 0.0
dev_1_p = 0.0
dev_1_e = 0.0
dev_2_p = 0.0
dev_2_e = 0.0

evolve_1PA = True
evolve_primary = False
evolve_2PA = False
deviation_included = True

p0 = 7.5

best_fit_0pa_vs_1pa= best_fit = [13.81557873,10.00059572,0.80005484,7.49973356,0.39998203,0.78521213, 1.0002002,0.95198958,0.33550665]

best_fit_with_deviation =  [1.38157362e+01,9.99767738e+00,8.00145324e-01,7.49920154e+00,3.99929753e-01,7.85259004e-01,1.00036356e+00,9.70707228e-01, 3.41813443e-01,4.30235889e-04,6.22299020e-04]

In [1]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from scipy.stats import chi2


from itertools import product
import os
import h5py
from tqdm import tqdm
import pandas as pd

#few utils
from few.utils.utility import get_p_at_t
from few.utils.constants import MTSUN_SI
from few.utils.geodesic import get_fundamental_frequencies
#few trajectory
from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode.flux import SuperKludgeFlux
#few waveform
from few.waveform import FastKerrEccentricEquatorialFlux, GenerateEMRIWaveform
from few.waveform.waveform import SuperKludgeWaveform
from few.utils.constants import YRSID_SI

#sef imports
from stableemrifisher.fisher import StableEMRIFisher
from stableemrifisher.utils import generate_PSD, padding, inner_product
from stableemrifisher.fisher.derivatives import derivative
from stableemrifisher.fisher.stablederivative import StableEMRIDerivative
from stableemrifisher.noise import sensitivity_LWA

#lisa-on-gpu import
from fastlisaresponse import ResponseWrapper  # Response function 

#LISAanalysistools imports
from lisatools.detector import ESAOrbits, EqualArmlengthOrbits #ESAOrbits correspond to esa-trailing-orbits.h5, EqualArmlengthOrbits are equalarmlength-orbits.h5
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens
from lisatools.sensitivity import get_sensitivity,CornishLISASens

use_gpu = True
from parismc.sampler import SamplerConfig
from parismc.sampler import Sampler

from smt.sampling_methods import LHS

if not use_gpu:
    
    import few
    
    #tune few configuration
    cfg_set = few.get_config_setter(reset=True)
    
    cfg_set.enable_backends("cpu")
    cfg_set.set_log_level("info");
else:
    pass #let the backend decide for itself

startup


In [4]:
#waveform class setup
waveform_class = SuperKludgeWaveform
max_step_days = 10.0 #max trajectory step size in days
inspiral_kwargs = {
    "err":1e-11, #default = 1e-11
    "max_step_size":max_step_days*24*60*60, #in seconds
}
sum_kwargs = {
    "pad_output": True, # True if expecting waveforms smaller than LISA observation window.
}

waveform_class_kwargs = dict(inspiral_kwargs=inspiral_kwargs,
                              mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
                              sum_kwargs=sum_kwargs,
                              use_gpu=use_gpu)

waveform_generator = GenerateEMRIWaveform
waveform_generator_kwargs = dict(return_list=False)


In [5]:
if(use_gpu):
    xp=cp
else:
    xp=np

best_fit_0pa_vs_1pa=  [13.81557873,10.00059572,0.80005484,7.49973356,0.39998203,0.78521213, 1.0002002,0.95198958,0.33550665]

best_fit_with_deviation = [1.38157362e+01,9.99767738e+00,8.00145324e-01,7.49920154e+00,3.99929753e-01,
                           7.85259004e-01,1.00036356e+00,9.70707228e-01, 3.41813443e-01,4.30235889e-04,6.22299020e-04]

#logm1_, m2_, a_, p0_, e0_,qS_,phiS_,Phi_phi0_,Phi_r0_,dev0p_,dev0e_ = params[i]

m1 = 1e6
m2 = 10
a = 0.8 # 0.95
e0 = 0.4 # 0.6 just spin first
xI0 = 1.0
dist = 0.4
qS = xp.pi/4
phiS = 1.0
qK = 1 
phiK = xp.pi/3
Phi_phi0 = 0.9
Phi_theta0 =0.5
Phi_r0 = 0.4

dt = 10.0
T = 1.0

chi2 = 0.0

dev_0_p=0.0
dev_0_e=0.0
dev_1_p=0.0
dev_1_e=0.0
dev_2_p=0.0
dev_2_e=0.0
evolve_1PA = False
evolve_primary = False
evolve_2PA = False
deviation_included=True
p0=7.5

print(use_gpu)
pars_list_com = [m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,\
             chi2,evolve_1PA,evolve_primary,evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

add_param_args={"chi2":chi2,"evolve_1PA":evolve_1PA,"evolve_primary":evolve_primary,"evolve_2PA":evolve_2PA,"deviation_included":deviation_included,"dev0p":dev_0_p,
"dev0e":dev_0_e,"dev1p":dev_1_p,"dev1e":dev_1_e,"dev2p":dev_2_p,"dev2e":dev_2_e}

param_names = ['m1','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0','dev0p','dev0e']
add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

emri_kwargs = {"T":T, "dt":dt}

param_names_com = ['m1','m2','a','p0','e0','xI0','dist','qS','phiS','qK','phiK','Phi_phi0','Phi_theta0','Phi_r0',"chi2",
               "evolve_1PA","evolve_primary","evolve_2PA","deviation_included","dev0p","dev0e","dev1p","dev1e","dev2p","dev2e"]

der_order = 8
Ndelta=16

True


## Fisher at Best FIT 0PA With Deviation

In [6]:
logm1_bf = best_fit_with_deviation[0]
m1_bf = np.exp(logm1_bf)
m2_bf = best_fit_with_deviation[1]
a_bf = best_fit_with_deviation[2]
p0_bf = best_fit_with_deviation[3]
e0_bf  = best_fit_with_deviation[4]
qS_bf = best_fit_with_deviation[5]
phiS_bf = best_fit_with_deviation[6]
Phi_phi0_bf = best_fit_with_deviation[7]
Phi_r0_bf = best_fit_with_deviation[8]
dev_0_p_bf =best_fit_with_deviation[9]
dev_0_e_bf =best_fit_with_deviation[10]

evolve_1PA_bf = False
deviation_included_bf = True

pars_list_com = [m1_bf, m2_bf, a_bf, p0_bf, e0_bf, xI0, dist, qS_bf, phiS_bf, qK, phiK, Phi_phi0_bf, Phi_theta0, Phi_r0_bf,\
             chi2,evolve_1PA_bf,evolve_primary,evolve_2PA,deviation_included_bf,dev_0_p_bf,dev_0_e_bf,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

add_param_args_bf ={"chi2":chi2,"evolve_1PA":evolve_1PA_bf,"evolve_primary":evolve_primary,"evolve_2PA":evolve_2PA,
                    "deviation_included":deviation_included_bf,"dev0p":dev_0_p_bf,"dev0e":dev_0_e_bf,
                    "dev1p":dev_1_p,"dev1e":dev_1_e,"dev2p":dev_2_p,"dev2e":dev_2_e}

param_names = ['m1','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0','dev0p','dev0e']

add_args_bf = [chi2, evolve_1PA_bf, evolve_primary, evolve_2PA,deviation_included_bf,dev_0_p_bf,dev_0_e_bf,dev_1_p,dev_1_e,dev_2_p,dev_2_e]


sef_bf = StableEMRIFisher(waveform_class=waveform_class, 
                       waveform_class_kwargs=waveform_class_kwargs,
                       waveform_generator=waveform_generator,
                       waveform_generator_kwargs=waveform_generator_kwargs,
                       stats_for_nerds = True, use_gpu = use_gpu,
                       deriv_type='stable',
                       noise_model=get_sensitivity,
                       noise_kwargs={'sens_fn':CornishLISASens,'return_type':'PSD'},
                       channels=["A","E"],
                       T = T, dt = dt,
                       stability_plot = False,
                       der_order = der_order, Ndelta = Ndelta,
                       plunge_check=True, return_derivatives=False
                       )
                   

SNR = sef_bf.SNRcalc_SEF(*pars_list_com,**emri_kwargs,use_gpu=use_gpu)
print("SNR: ", SNR)

param_dict_bf = {
    'm1': m1_bf,
    'm2': m2_bf,
    'a': a_bf,
    'p0': p0_bf,
    'e0': e0_bf,
    'xI0': xI0,
    'dist': dist,
    'qS': qS_bf,
    'phiS': phiS_bf,
    'qK': qK,
    'phiK': phiK,
    'Phi_phi0': Phi_phi0_bf,
    'Phi_theta0': Phi_theta0,
    'Phi_r0': Phi_r0_bf
}


Fisher_bf = sef_bf(wave_params = param_dict_bf,param_names=param_names, add_param_args=add_param_args_bf,
            live_dangerously = False, stability_plot = True,der_order = der_order, Ndelta = Ndelta,return_derivatives=False
            )

OutOfMemoryError: Out of memory allocating 121,846,272 bytes (allocated so far: 5,090,943,488 bytes).

## Fisher Best FIT Point 0PA without Deviation

In [ ]:
logm1_0v1_bf = best_fit_0pa_vs_1pa[0]
m1_0v1_bf = np.exp(logm1_0v1_bf)
m2_0v1_bf = best_fit_0pa_vs_1pa[1]
a_0v1_bf = best_fit_0pa_vs_1pa[2]
p0_0v1_bf = best_fit_0pa_vs_1pa[3]
e0_0v1_bf  = best_fit_0pa_vs_1pa[4]
qS_0v1_bf = best_fit_0pa_vs_1pa[5]
phiS_0v1_bf = best_fit_0pa_vs_1pa[6]
Phi_phi0_0v1_bf = best_fit_0pa_vs_1pa[7]
Phi_r0_0v1_bf = best_fit_0pa_vs_1pa[8]

evolve_1PA_0v1_bf = False
deviation_included_0v1_bf = True

pars_list_com = [m1_0v1_bf, m2_0v1_bf, a_0v1_bf, p0_0v1_bf, e0_0v1_bf, xI0, dist, qS_0v1_bf, phiS_0v1_bf, qK,
                  phiK, Phi_phi0_0v1_bf, Phi_theta0, Phi_r0_0v1_bf,chi2,evolve_1PA_0v1_bf,evolve_primary,
                  evolve_2PA,deviation_included_0v1_bf,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

add_param_args_0v1_bf ={"chi2":chi2,"evolve_1PA":evolve_1PA_0v1_bf,"evolve_primary":evolve_primary,"evolve_2PA":evolve_2PA,
                    "deviation_included":deviation_included_0v1_bf,"dev0p":dev_0_p,"dev0e":dev_0_e,
                    "dev1p":dev_1_p,"dev1e":dev_1_e,"dev2p":dev_2_p,"dev2e":dev_2_e}

param_names = ['m1','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0']

add_args_0v1_bf = [chi2, evolve_1PA_0v1_bf, evolve_primary, evolve_2PA,deviation_included_0v1_bf,
                   dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]


sef_0v1_bf = StableEMRIFisher(waveform_class=waveform_class, 
                       waveform_class_kwargs=waveform_class_kwargs,
                       waveform_generator=waveform_generator,
                       waveform_generator_kwargs=waveform_generator_kwargs,
                       stats_for_nerds = True, use_gpu = use_gpu,
                       deriv_type='stable',
                       noise_model=get_sensitivity,
                       noise_kwargs={'sens_fn':CornishLISASens,'return_type':'PSD'},
                       channels=["A","E"],
                       T = T, dt = dt,
                       stability_plot = False,
                       der_order = der_order, Ndelta = Ndelta,
                       plunge_check=True, return_derivatives=False
                       )
                   

SNR = sef_0v1_bf.SNRcalc_SEF(*pars_list_com,**emri_kwargs,use_gpu=use_gpu)
print("SNR: ", SNR)

param_dict_0v1_bf = {
    'm1': m1_0v1_bf,
    'm2': m2_0v1_bf,
    'a': a_0v1_bf,
    'p0': p0_0v1_bf,
    'e0': e0_0v1_bf,
    'xI0': xI0,
    'dist': dist,
    'qS': qS_0v1_bf,
    'phiS': phiS_0v1_bf,
    'qK': qK,
    'phiK': phiK,
    'Phi_phi0': Phi_phi0_0v1_bf,
    'Phi_theta0': Phi_theta0,
    'Phi_r0': Phi_r0_0v1_bf
}


Fisher_0v1_bf = sef_0v1_bf(wave_params = param_dict_0v1_bf,param_names=param_names, add_param_args=add_param_args_0v1_bf,
            live_dangerously = False, stability_plot = True,der_order = der_order, Ndelta = Ndelta,return_derivatives=False
            )

## Fisher True Point 1PA

In [ ]:
evolve_1PA = True
pars_list_com = [m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,\
             chi2,evolve_1PA,evolve_primary,evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

add_param_args={"chi2":chi2,"evolve_1PA":evolve_1PA,"evolve_primary":evolve_primary,"evolve_2PA":evolve_2PA,"deviation_included":deviation_included,"dev0p":dev_0_p,
"dev0e":dev_0_e,"dev1p":dev_1_p,"dev1e":dev_1_e,"dev2p":dev_2_p,"dev2e":dev_2_e}

param_names = ['m1','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0']

add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]


sef_true = StableEMRIFisher(waveform_class=waveform_class, 
                       waveform_class_kwargs=waveform_class_kwargs,
                       waveform_generator=waveform_generator,
                       waveform_generator_kwargs=waveform_generator_kwargs,
                       stats_for_nerds = True, use_gpu = use_gpu,
                       deriv_type='stable',
                       noise_model=get_sensitivity,
                       noise_kwargs={'sens_fn':CornishLISASens,'return_type':'PSD'},
                       channels=["A","E"],
                       T = T, dt = dt,
                       stability_plot = False,
                       der_order = der_order, Ndelta = Ndelta,
                       plunge_check=True, return_derivatives=False
                       )
                   

SNR = sef_true.SNRcalc_SEF(*pars_list_com,**emri_kwargs,use_gpu=use_gpu)

pars_list = [m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]

param_dict = {
    'm1': m1,
    'm2': m2,
    'a': a,
    'p0': p0,
    'e0': e0,
    'xI0': xI0,
    'dist': dist,
    'qS': qS,
    'phiS': phiS,
    'qK': qK,
    'phiK': phiK,
    'Phi_phi0': Phi_phi0,
    'Phi_theta0': Phi_theta0,
    'Phi_r0': Phi_r0
}


Fisher_true = sef_true(wave_params = param_dict,param_names=param_names, add_param_args=add_param_args,
            live_dangerously = False, stability_plot = True,der_order = der_order, Ndelta = Ndelta,return_derivatives=False
            )

In [ ]:
# def logmasstransform(Fisher, m1, index_of_m1 = 0):    
#     J = np.eye(len(Fisher))
#     J[index_of_m1,index_of_m1] = m1
    
#     return J.T@Fisher@J
param_true= [m1, m2, a, p0, e0,qS,phiS,Phi_phi0,Phi_r0,dev_0_p,dev_0_e] 
param_0v1_bf = best_fit_0pa_vs_1pa.copy()
param_0v1_bf[0] = np.exp(param_0v1_bf[0])
param_bf = best_fit_with_deviation.copy()
param_bf[0] = np.exp(param_bf[0])

Cov_bf = np.linalg.inv(Fisher_bf)
Cov_0v1_bf = np.linalg.inv(Fisher_0v1_bf)
Cov_true = np.linalg.inv(Fisher_true)

In [ ]:
Cov_0v1_bf

In [ ]:
# FOR FIRST EIGHT PARAMETERS ONLY (DOES NOT INCLUDE DEVIATION PARAMETERS)
from scipy.stats import norm

npar = 8
for i in range(npar):

    mu1, sigma1 = param_true[i], np.sqrt(Cov_true[i, i])
    mu2, sigma2 = param_bf[i], np.sqrt(Cov_bf[i, i])
    mu3, sigma3 = param_0v1_bf[i], np.sqrt(Cov_0v1_bf[i, i])

    print(sigma1,sigma2,sigma3)

    # x-range covering all three posteriors
    xmin = min(mu1 - 3*sigma1, mu2 - 3*sigma2, mu3 - 3*sigma3)
    xmax = max(mu1 + 3*sigma1, mu2 + 3*sigma2, mu3 + 3*sigma3)
    x = np.linspace(xmin, xmax, 1000)

    plt.figure()

    line1, = plt.plot(x, norm.pdf(x, mu1, sigma1), label="Value at True Point")
    line2, = plt.plot(x, norm.pdf(x, mu2, sigma2), label="Best Fit With Deviation")
    line3, = plt.plot(x, norm.pdf(x, mu3, sigma3), label="Best Fit Without Deviation")

    plt.axvline(mu1, linestyle="--", color=line1.get_color())
    plt.axvline(mu2, linestyle="--", color=line2.get_color())
    plt.axvline(mu3, linestyle="--", color=line3.get_color())

    plt.xlabel(param_names[i])
    plt.ylabel("Marginalized Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
npar = 8
for i in range(npar+1,11):

    mu1, sigma1 = param_true[i], np.sqrt(Cov_true[i, i])
    mu2, sigma2 = param_bf[i], np.sqrt(Cov_bf[i, i])

    # x-range covering all three posteriors
    xmin = min(mu1 - 4*sigma1, mu2 - 4*sigma2)
    xmax = max(mu1 + 4*sigma1, mu2 + 4*sigma2)
    x = np.linspace(xmin, xmax, 500)

    plt.figure()


    line1, = plt.plot(x, norm.pdf(x, mu1, sigma1), label="Value at True Point")
    line2, = plt.plot(x, norm.pdf(x, mu2, sigma2), label="Best Fit With Deviation")

    plt.axvline(mu1, linestyle="--", color=line1.get_color())
    plt.axvline(mu2, linestyle="--", color=line2.get_color())


    plt.xlabel(param_names[i])
    plt.ylabel("Marginalized Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
mu1, sigma1 = param_true[1], np.sqrt(Cov_true[1, 1])
mu2, sigma2 = param_bf[1], np.sqrt(Cov_bf[1, 1])
mu3, sigma3 = param_0v1_bf[1], np.sqrt(Cov_0v1_bf[1, 1])
print(sigma1,sigma2,sigma3)
# x-range covering all three posteriors
xmin = min(mu1 - 10*sigma1, mu2 - 10*sigma2, mu3 - 10*sigma3)
xmax = max(mu1 + 10*sigma1, mu2 + 10*sigma2, mu3 + 10*sigma3)
x = np.linspace(xmin, xmax, 1000)
plt.figure()

line1, = plt.plot(x, norm.pdf(x, mu1, sigma1), label="Value at True Point")
line2, = plt.plot(x, norm.pdf(x, mu2, sigma2), label="Best Fit With Deviation")
line3, = plt.plot(x, norm.pdf(x, mu3, sigma3), label="Best Fit Without Deviation")
plt.axvline(mu1, linestyle="--", color=line1.get_color())
plt.axvline(mu2, linestyle="--", color=line2.get_color())
plt.axvline(mu3, linestyle="--", color=line3.get_color())

plt.xlabel(param_names[1])
plt.ylabel("Marginalized Density")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
mu1, sigma1 = param_true[1], np.sqrt(Cov_true[1, 1])
mu2, sigma2 = param_bf[1], np.sqrt(Cov_bf[1, 1])
mu3, sigma3 = param_0v1_bf[1], np.sqrt(Cov_0v1_bf[1, 1])
print(sigma1,sigma2,sigma3)
# x-range covering all three posteriors
xmin = min(mu1 - 0.1*sigma1, mu2 - 0.1*sigma2, mu3 - 0.1*sigma3)
xmax = max(mu1 + 0.1*sigma1, mu2 + 0.1*sigma2, mu3 + 0.1*sigma3)
x = np.linspace(xmin, xmax, 1000)
plt.figure()

line1, = plt.plot(x, norm.pdf(x, mu1, sigma1), label="Value at True Point")
line2, = plt.plot(x, norm.pdf(x, mu2, sigma2), label="Best Fit With Deviation")
line3, = plt.plot(x, norm.pdf(x, mu3, sigma3), label="Best Fit Without Deviation")
plt.axvline(mu1, linestyle="--", color=line1.get_color())
plt.axvline(mu2, linestyle="--", color=line2.get_color())
plt.axvline(mu3, linestyle="--", color=line3.get_color())

plt.xlabel(param_names[1])
plt.ylabel("Marginalized Density")
plt.legend()
plt.tight_layout()
plt.show()